# Residual Attention U-Net — Ablation Study

Runs 3 incremental configurations on Kaggle GPU and produces:
- Per-run evaluation outputs (metrics, confusion matrices, predictions, boundary plots)
- Combined `ablation_comparison.csv` with side-by-side results

| Version | Input | Loss | LR Schedule |
|:---:|---|---|---|
| 1 (Baseline) | 4-Bands (RGB+NIR) | Dice + Focal | Plateau |
| 2 (Edge Loss) | 4-Bands (RGB+NIR) | Boundary-Weighted Focal | Plateau |
| 3 (**Proposed**) | 6-Bands (RGB+NIR+NDVI+NDWI) | Boundary-Weighted Focal | Cosine Annealing |

> **Before running:** Set your GitHub repo URL in the cell below.

In [ ]:
GITHUB_REPO_URL = "https://github.com/ATIK2110018/semantic_segmentation.git"
REPO_DIR = "/kaggle/working/segment"
OUTPUT_BASE = "/kaggle/working/ablation_results"

In [ ]:
import os
if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest...")
    os.system(f"git -C {REPO_DIR} pull")
else:
    print("Cloning repo...")
    os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
print("Done.")

In [ ]:
os.system(f"pip install -q -r {REPO_DIR}/requirements.txt")
print("Dependencies installed.")

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {gpus}")
if not gpus:
    print("WARNING: No GPU detected. Go to Settings → Accelerator → GPU T4 x2")

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, f"{REPO_DIR}/run_ablation.py",
    "--data_path",        f"{REPO_DIR}/dataset",
    "--patch_size",       "256",
    "--patch_step",       "128",
    "--epochs",           "200",
    "--batch_size",       "16",
    "--lr",               "1e-4",
    "--patience",         "30",
    "--num_samples",      "5",
    "--ablation_output",  OUTPUT_BASE,
]

print(f"Command: {' '.join(cmd)}\n")
process = subprocess.run(cmd, cwd=REPO_DIR)
print(f"\nExit code: {process.returncode}")

## Ablation Comparison Table

In [ ]:
import pandas as pd
from IPython.display import display

csv_path = f"{OUTPUT_BASE}/ablation_comparison.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)
else:
    print(f"Comparison CSV not found at {csv_path}")

## Training History (All Runs)

In [ ]:
from IPython.display import Image, display, Markdown
import glob

runs = sorted(glob.glob(f"{OUTPUT_BASE}/*/"))
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    history_path = os.path.join(run_dir, "training_history.png")
    if os.path.exists(history_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=history_path, width=800))

## Confusion Matrices

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    cm_path = os.path.join(run_dir, "confusion_matrix.png")
    if os.path.exists(cm_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=cm_path, width=900))

## Per-Class IoU Charts

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    iou_path = os.path.join(run_dir, "per_class_iou.png")
    if os.path.exists(iou_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=iou_path, width=800))

## All Metrics Charts

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    metrics_path = os.path.join(run_dir, "all_metrics_chart.png")
    if os.path.exists(metrics_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=metrics_path, width=900))

## Prediction Samples

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    pred_path = os.path.join(run_dir, "predictions.png")
    if os.path.exists(pred_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=pred_path, width=900))

## Boundary Predictions (Color-Coded)

- 🟢 **Green** = Correct boundary (TP)
- 🔴 **Red** = Missed boundary (FN)
- 🟡 **Yellow** = False boundary (FP)

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    bnd_path = os.path.join(run_dir, "boundary_predictions.png")
    if os.path.exists(bnd_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=bnd_path, width=1000))

## Boundary Metrics Charts

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    chart_path = os.path.join(run_dir, "boundary_metrics_chart.png")
    if os.path.exists(chart_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=chart_path, width=900))

## Per-Run Evaluation CSVs

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    eval_csv = os.path.join(run_dir, "evaluation_results.csv")
    bnd_csv = os.path.join(run_dir, "boundary_results_global.csv")
    
    if os.path.exists(eval_csv):
        display(Markdown(f"### {run_name} — Pixel Metrics"))
        display(pd.read_csv(eval_csv))
    if os.path.exists(bnd_csv):
        display(Markdown(f"### {run_name} — Boundary Metrics"))
        display(pd.read_csv(bnd_csv))

## Class Legend

In [ ]:
legend_path = None
for run_dir in runs:
    p = os.path.join(run_dir, "class_legend.png")
    if os.path.exists(p):
        legend_path = p
        break
if legend_path:
    display(Image(filename=legend_path, width=400))

## Download Results

Run the cell below to zip all outputs into a single downloadable file.

In [ ]:
import shutil
from IPython.display import FileLink

shutil.make_archive('/kaggle/working/ablation_results', 'zip', OUTPUT_BASE)
print("Zip created: ablation_results.zip")
display(FileLink('ablation_results.zip'))

In [ ]:
for run_dir in sorted(glob.glob(f"{OUTPUT_BASE}/*/")):
    run_name = os.path.basename(run_dir.rstrip('/'))
    files = os.listdir(run_dir)
    print(f"\n{run_name}/ ({len(files)} files)")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(run_dir, f))
        print(f"  {f:.<45} {size/1024:.1f} KB")

comp = f"{OUTPUT_BASE}/ablation_comparison.csv"
if os.path.exists(comp):
    print(f"\nablation_comparison.csv .... {os.path.getsize(comp)/1024:.1f} KB")